# Évaluation V5-mini — Phases 6 / 7 / 8

**Notebook unique** pour évaluer la version Oracle V5-mini contre le baseline CorrDiff noncausal.

**Trois phases successives** :

1. **Phase 6 — Comparaison in-distribution standardisée**
   Tableau métrique × variant sur le jeu de test historique. Charge les JSON `final_validation_metrics.json` produits par le notebook de training.

2. **Phase 7 — Évaluation hors-distribution (OOD)**
   Si un dataset SSP5-8.5 ou EC-Earth3 est dispo : compare Δ_OOD. Sinon : diagnostics de cohérence physique (RAPSD, distribution centiles).

3. **Phase 8 — Protocole d'intervention `do(·)`**
   Charge les modèles et applique 3 interventions standardisées sur l'humidité, la température, le vent. Mesure Q_int.

**Pré-requis** :
- Checkpoints sur Drive :
  - V5-mini : `/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal/`
  - Noncausal : `/content/drive/MyDrive/climate_data/ckpt_noncausal/`
- Phase 6 ne nécessite pas le rechargement des modèles (post-hoc).
- Phase 8 nécessite le rechargement complet.

**Sortie** : `/content/drive/MyDrive/climate_data/results/v5_evaluation/`

In [ ]:
# Setup Colab : mount Drive + clone repo si nécessaire
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
print(f"On Colab : {ON_COLAB}")

if ON_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

    REPO_DIR = Path("/content/climate_data")
    if not REPO_DIR.exists():
        os.system("git clone https://github.com/leonelkenfack/climate_data.git /content/climate_data")
    os.chdir(REPO_DIR)

REPO_ROOT = Path(os.getcwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT : {REPO_ROOT}")

In [ ]:
# === CONFIGURATION DES CHEMINS ===
from pathlib import Path

# V5-mini (directory historiquement nommé ckpt_v2_corrdiff_normal)
V5_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal")

# Noncausal = baseline CorrDiff générique
NONCAUSAL_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_noncausal")

# Sorties
RESULTS_DIR = Path("/content/drive/MyDrive/climate_data/results/v5_evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Vérification existence
for label, p in [("V5", V5_DIR), ("Noncausal", NONCAUSAL_DIR)]:
    ckpt = p / "epoch_last.pth"
    fv = p / "final_validation_metrics.json"
    status_ckpt = "OK" if ckpt.exists() else "ABSENT"
    status_fv = "OK" if fv.exists() else "ABSENT"
    size_gb = ckpt.stat().st_size / 1e9 if ckpt.exists() else 0
    print(f"  {label:10s} : ckpt {status_ckpt} ({size_gb:.2f} GB)  metrics.json {status_fv}")
    print(f"             {p}")

print(f"\nRésultats -> {RESULTS_DIR}")

---

## Phase 6 — Comparaison in-distribution standardisée

Lit `final_validation_metrics.json` de chaque variante et produit un tableau métrique × variant. **Pas de rechargement des modèles**, post-hoc.

Métriques comparées :
- Pearson global + per-sample
- RMSE / MAE / Spread / Spread-RMSE ratio
- F1-p95 / F1-p99 (Pearson restreint aux centiles)
- RAPSD distance
- μ_HR ablation ratio (Δ_O3 architectural)

In [ ]:
# Phase 6 : comparaison in-distribution post-hoc
import json
import numpy as np

def load_metrics(d):
    p = Path(d) / "final_validation_metrics.json"
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8"))

m_v5 = load_metrics(V5_DIR)
m_nc = load_metrics(NONCAUSAL_DIR)

if m_v5 is None:
    print(f"[ERREUR] V5 metrics absent : {V5_DIR}/final_validation_metrics.json")
if m_nc is None:
    print(f"[ERREUR] Noncausal metrics absent : {NONCAUSAL_DIR}/final_validation_metrics.json")

if m_v5 and m_nc:
    def get(m, *path, default=None):
        v = m
        for k in path:
            if not isinstance(v, dict) or k not in v:
                return default
            v = v[k]
        return v

    rows = [
        ("Pearson global",     get(m_v5, "pearson_corr", "global"),         get(m_nc, "pearson_corr", "global"),         "haut"),
        ("Pearson per-sample", get(m_v5, "pearson_corr", "per_sample_avg"), get(m_nc, "pearson_corr", "per_sample_avg"), "haut"),
        ("RMSE",               get(m_v5, "rmse"),                            get(m_nc, "rmse"),                            "bas"),
        ("MAE",                get(m_v5, "mae"),                             get(m_nc, "mae"),                             "bas"),
        ("Spread (ens std)",   get(m_v5, "spread_mean"),                     get(m_nc, "spread_mean"),                     "calib"),
        ("F1-p95",             get(m_v5, "f1_extremes", "p95"),              get(m_nc, "f1_extremes", "p95"),              "haut"),
        ("F1-p99",             get(m_v5, "f1_extremes", "p99"),              get(m_nc, "f1_extremes", "p99"),              "haut"),
        ("RAPSD distance",     get(m_v5, "rapsd_distance"),                  get(m_nc, "rapsd_distance"),                  "bas"),
        ("mu_HR ablation",     get(m_v5, "mu_HR_ablation", "delta_signal_ratio_avg"), get(m_nc, "mu_HR_ablation", "delta_signal_ratio_avg"), "haut"),
    ]

    sr_v5 = (rows[4][1] or 0) / (rows[2][1] or 1)
    sr_nc = (rows[4][2] or 0) / (rows[2][2] or 1)
    rows.insert(5, ("Spread/RMSE", sr_v5, sr_nc, "vers 1"))

    print(f"\n{'Metrique':<22} {'V5-mini':>12} {'Noncausal':>12} {'D abs':>10} {'D rel %':>10} {'sens':>8} {'gagnant':>10}")
    print("-" * 100)
    summary = {}
    for name, v5, nc, sens in rows:
        if v5 is None or nc is None:
            print(f"{name:<22} {'n/a':>12} {'n/a':>12}")
            continue
        d_abs = v5 - nc
        d_rel = 100 * d_abs / nc if abs(nc) > 1e-12 else float('nan')
        if abs(d_rel) < 1.0:
            winner = "egal"
        elif sens == "haut" and d_abs > 0:
            winner = "V5"
        elif sens == "bas" and d_abs < 0:
            winner = "V5"
        elif sens == "calib":
            winner = "V5" if d_abs > 0 else "Noncausal"
        elif sens == "vers 1":
            winner = "V5" if abs(v5 - 1) < abs(nc - 1) else "Noncausal"
        else:
            winner = "Noncausal"
        summary[name] = {"v5": v5, "noncausal": nc, "delta_abs": d_abs, "delta_rel_pct": d_rel, "winner": winner}
        print(f"{name:<22} {v5:>12.4f} {nc:>12.4f} {d_abs:>+10.4f} {d_rel:>+10.2f} {sens:>8} {winner:>10}")

    out = RESULTS_DIR / "phase6_in_distribution.json"
    out.write_text(json.dumps({
        "v5_dir": str(V5_DIR),
        "noncausal_dir": str(NONCAUSAL_DIR),
        "comparison": summary,
        "raw_v5_metrics": m_v5,
        "raw_noncausal_metrics": m_nc,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n[OK] Phase 6 sauvegardee : {out}")

---

## Phase 7 — Évaluation OOD (changement de régime)

Deux modes :

### Mode A — Dataset OOD disponible
Définis `OOD_DATA_PATH` ci-dessous (ex. `EC-Earth3_histupdated_compressed.nc`) et lance.

### Mode B — Fallback (pas de dataset OOD)
Si pas de dataset, on calcule des diagnostics de cohérence physique à partir des `final_validation_metrics.json` :
- Spread/RMSE (calibration probabiliste)
- RAPSD distance (structure spectrale)
- μ_HR ablation (dépendance causale architecturale)
- Shortcut ratio (qualité diffusion)

Le mode B ne mesure pas Δ_OOD strict mais donne un argument qualitatif sur la robustesse structurelle.

In [ ]:
# Phase 7 : OOD test (mode A si data dispo, mode B fallback sinon)
import json

# === Configure ici si tu as un dataset OOD ===
OOD_DATA_PATH = None  # ex: Path("/content/data_local/test/EC-Earth3_histupdated_compressed.nc")
# =============================================

if OOD_DATA_PATH is not None and Path(OOD_DATA_PATH).exists():
    print(f"Mode A : OOD dataset detecte ({OOD_DATA_PATH})")
    print("Pour eval OOD complet :")
    print("  - Surcharger CONFIG.data.hr_path = OOD_DATA_PATH")
    print("  - Relancer st_cdgm_validation_inference.ipynb avec CKPT_SAVE_DIR pointant V5_DIR")
    print("  - Sauvegarder les metrics OOD dans V5_DIR/final_validation_metrics_OOD.json")
    print("  - Idem pour noncausal")

else:
    print("Mode B : diagnostics de coherence physique (post-hoc)")
    print()

    m_v5 = json.loads((V5_DIR / "final_validation_metrics.json").read_text(encoding="utf-8"))
    m_nc = json.loads((NONCAUSAL_DIR / "final_validation_metrics.json").read_text(encoding="utf-8"))

    print(f"{'Diagnostic':<32} {'V5-mini':>12} {'Noncausal':>12} {'Interpretation':<35}")
    print("-" * 100)

    sr_v5 = m_v5["spread_mean"] / m_v5["rmse"]
    sr_nc = m_nc["spread_mean"] / m_nc["rmse"]
    print(f"{'Spread/RMSE (calibration)':<32} {sr_v5:>12.4f} {sr_nc:>12.4f} {'ideal=1, sub-disp si <1':<35}")
    print(f"{'RAPSD distance':<32} {m_v5['rapsd_distance']:>12.4f} {m_nc['rapsd_distance']:>12.4f} {'plus bas = mieux':<35}")

    ab_v5 = m_v5["mu_HR_ablation"]["delta_signal_ratio_avg"]
    ab_nc = m_nc["mu_HR_ablation"]["delta_signal_ratio_avg"]
    print(f"{'mu_HR ablation D/signal':<32} {ab_v5:>12.4f} {ab_nc:>12.4f} {'plus haut = plus causal':<35}")

    sc_v5 = m_v5["shortcut_diagnostic"]["shortcut_ratio"]
    sc_nc = m_nc["shortcut_diagnostic"]["shortcut_ratio"]
    print(f"{'Shortcut ratio':<32} {sc_v5:>12.4f} {sc_nc:>12.4f} {'>1 = vraie diffusion':<35}")

    print()
    print("Verdict qualitatif :")
    if ab_v5 > ab_nc:
        print(f"  [+] V5-mini : dependance mu_HR plus forte (+{(ab_v5-ab_nc)*100:.1f}%) -> causalite structurelle")
    if sr_v5 > sr_nc:
        print(f"  [+] V5-mini : meilleure calibration probabiliste (+{(sr_v5-sr_nc)*100:.1f}%)")
    if m_v5["rapsd_distance"] < m_nc["rapsd_distance"]:
        delta_rapsd = (m_nc['rapsd_distance']-m_v5['rapsd_distance'])/m_nc['rapsd_distance']*100
        print(f"  [+] V5-mini : meilleure fidelite spectrale (-{delta_rapsd:.1f}%)")
    print()
    print("[INFO] D_OOD strict non mesure (pas de dataset OOD pointe)")

    out = RESULTS_DIR / "phase7_ood_physical_diagnostics.json"
    out.write_text(json.dumps({
        "mode": "physical_diagnostics_fallback",
        "v5": {
            "spread_rmse_ratio": sr_v5,
            "rapsd_distance": m_v5["rapsd_distance"],
            "mu_HR_ablation": ab_v5,
            "shortcut_ratio": sc_v5,
        },
        "noncausal": {
            "spread_rmse_ratio": sr_nc,
            "rapsd_distance": m_nc["rapsd_distance"],
            "mu_HR_ablation": ab_nc,
            "shortcut_ratio": sc_nc,
        },
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n[OK] Phase 7 (fallback) sauvegardee : {out}")

---

## Phase 8 — Protocole d'intervention `do(·)`

Cette phase **charge les modèles** et applique 3 interventions standardisées :

1. `do(q_850 × 1.20)` — humidité ↑20 % → précipitation devrait ↑
2. `do(t_850 += 3 K)` — réchauffement → précipitation devrait ↑
3. `do(u_850 × 1.10)` — vent ↑10 % → effet attendu marqué (advection humide)

Pour chaque intervention : `Δ_pred = predict(intervened) - predict(normal)` confronté au signe physiquement attendu.

**Q_int** = fraction des interventions où le signe correspond à l'attendu.

In [ ]:
# Phase 8 : Intervention test
# Cette cellule charge les helpers et liste les interventions standardisées.
# L'exécution complète nécessite de charger les deux modèles.

from scripts.intervention_test import (
    INTERVENTIONS,
    resolve_variable_indices,
    apply_intervention,
    evaluate_intervention,
    compute_q_int,
    save_intervention_results,
)

print("Interventions standardisees :")
for spec in INTERVENTIONS:
    print(f"  - {spec['name']:<28s}  {spec['variable_name']}: "
          f"{spec['delta_type']} {spec['delta_value']:>6}  attendu {spec['expected_sign']:+d}")
print()

# Résolution des indices de variables LR depuis la config
# (Nécessite que CONFIG soit chargé — typiquement fait dans le notebook training_evaluation)
print("Pour executer Phase 8, dans une cellule suivante :")
print()
print("  1. Charger CONFIG (cf. notebook training_evaluation cell 14-20)")
print("  2. Recharger encoder + rcn_runner + regression_head + diffusion depuis V5_DIR")
print("     (puis a nouveau depuis NONCAUSAL_DIR)")
print("  3. Definir une fonction predict(batch) qui retourne l'ensemble [K, B, 1, H, W]")
print("  4. Appeler les helpers de scripts.intervention_test :")
print()
print("Exemple de boucle (a adapter) :")
print('-' * 60)

In [ ]:
# ============================================================
# Phase 8 - EXECUTION COMPLETE
# Charge les 2 modeles, evalue 3 interventions, sauvegarde Q_int.
#
# Pre-requis :
#   - CONFIG charge (depuis training_evaluation cell 14-20)
#   - builder, test_dataset prepares
#   - DEVICE (typiquement torch.device('cuda'))
#   - convert_sample_to_batch defini
#
# Si pas encore charge : execute d'abord les cellules 14 a 30 de
#   st_cdgm_training_evaluation.ipynb puis reviens ici.
# ============================================================

import json
import time
import torch
import numpy as np

# 0. Verification des pre-requis
required_globals = ['CONFIG', 'builder', 'DEVICE', 'convert_sample_to_batch']
missing = [g for g in required_globals if g not in globals()]
if missing:
    raise RuntimeError(
        f"Globals manquants : {missing}. "
        "Execute d'abord les cellules d'init de st_cdgm_training_evaluation.ipynb."
    )

print(f"[OK] Pre-requis valides : {required_globals}")
print(f"     DEVICE = {DEVICE}")
print(f"     lr_variables = {list(CONFIG.data.lr_variables)}")
print()

# ── 1. Helpers de chargement d'un stack complet depuis un checkpoint ─────
from st_cdgm.models import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    GraphToGridDecoder, RCNCell, RCNSequenceRunner,
    CausalDiffusionDecoder,
)
try:
    from st_cdgm.models import ConditionalSkipBlock
    SKIP_AVAILABLE = True
except ImportError:
    SKIP_AVAILABLE = False
    print("[INFO] ConditionalSkipBlock non disponible (V5 pas encore deploye)")

def build_stack_from_ckpt(ckpt_path, variant_name):
    """Construit encoder + RCN + regression_head + (skip si V5) + diffusion
    et charge les poids depuis ckpt_path."""
    print(f"  [{variant_name}] Loading {ckpt_path}")
    t0 = time.time()
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    print(f"  [{variant_name}] Checkpoint loaded in {time.time()-t0:.1f}s "
          f"(epoch={ckpt.get('epoch')}, causal_concat={ckpt.get('causal_concat')})")

    enc_cfg = IntelligibleVariableConfig(
        num_variables=len(CONFIG.data.lr_variables),
        hidden_dim=CONFIG.encoder.hidden_dim,
        conditioning_dim=CONFIG.encoder.conditioning_dim,
        num_dag_tokens=int(CONFIG.encoder.get("num_dag_tokens", 2)),
        causal_conditioning=bool(CONFIG.encoder.get("causal_conditioning", True)),
    )
    encoder = IntelligibleVariableEncoder(enc_cfg).to(DEVICE)

    rcn_cell = RCNCell(
        num_variables=enc_cfg.num_variables,
        hidden_dim=CONFIG.rcn.hidden_dim,
        driver_dim=CONFIG.rcn.driver_dim,
        reconstruction_dim=CONFIG.rcn.reconstruction_dim,
        dropout=CONFIG.rcn.dropout,
    ).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.detach_interval)

    regression_head = GraphToGridDecoder(
        d_model=CONFIG.encoder.hidden_dim,
        hr_h=CONFIG.graph.hr_shape[0],
        hr_w=CONFIG.graph.hr_shape[1],
    ).to(DEVICE)

    from st_cdgm.models.edm_preconditioner import EDMConfig
    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
    diffusion = CausalDiffusionDecoder(
        in_channels=CONFIG.diffusion.in_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=CONFIG.diffusion.height,
        width=CONFIG.diffusion.width,
        scheduler_type=CONFIG.diffusion.scheduler_type,
        causal_concat=True,
        edm_config=edm_cfg,
        unet_kwargs=dict(CONFIG.diffusion.unet_kwargs),
    ).to(DEVICE)

    def _load(name, module):
        key = f"{name}_state_dict"
        if key not in ckpt:
            print(f"  [{variant_name}] [WARN] {key} absent")
            return False
        module.load_state_dict(ckpt[key])
        return True
    _load("encoder", encoder)
    _load("rcn_cell", rcn_cell)
    _load("regression_head", regression_head)
    _load("diffusion", diffusion)

    skip_block = None
    if SKIP_AVAILABLE and "skip_block_state_dict" in ckpt:
        skip_block = ConditionalSkipBlock(
            lr_channels=len(CONFIG.data.lr_variables),
            hr_shape=tuple(CONFIG.graph.hr_shape),
        ).to(DEVICE)
        skip_block.load_state_dict(ckpt["skip_block_state_dict"])
        print(f"  [{variant_name}] [+] ConditionalSkipBlock loaded ({skip_block.num_params()} params)")

    encoder.eval(); rcn_cell.eval(); regression_head.eval(); diffusion.eval()
    if skip_block is not None:
        skip_block.eval()

    return {
        "encoder": encoder, "rcn_runner": rcn_runner,
        "regression_head": regression_head, "diffusion": diffusion,
        "skip_block": skip_block,
        "variant": variant_name,
    }

# ── 2. Chargement des deux stacks ──────────────────────────────────
print("Chargement des 2 stacks (V5 + Noncausal)...")
print()
t_load = time.time()
stack_v5 = build_stack_from_ckpt(V5_DIR / "epoch_last.pth", "V5")
print()
stack_nc = build_stack_from_ckpt(NONCAUSAL_DIR / "epoch_last.pth", "Noncausal")
print()
print(f"[OK] 2 stacks charges en {time.time()-t_load:.1f}s")
print()

# ── 3. Fonction predict generique ──────────────────────────────────
@torch.no_grad()
def predict_with_stack(stack, batch, K=4, n_steps=32):
    """Genere un ensemble [K, B, 1, H, W] depuis un stack + batch."""
    encoder = stack["encoder"]
    rcn_runner = stack["rcn_runner"]
    regression_head = stack["regression_head"]
    diffusion = stack["diffusion"]
    skip_block = stack["skip_block"]

    lr_data = batch["lr"].to(DEVICE)
    H_init = encoder.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(H_init, drivers, reconstruction_sources=None)
    H_T = seq_out.states[-1]
    mu_HR_causal = regression_head(H_T)

    target_shape = batch["residual"][-1].to(DEVICE).shape
    if target_shape[-2:] != mu_HR_causal.shape[-2:]:
        mu_HR_causal = torch.nn.functional.interpolate(
            mu_HR_causal, size=target_shape[-2:],
            mode="bilinear", align_corners=False,
        )
    if skip_block is not None:
        lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
        mu_HR, alpha = skip_block(lr_last, mu_HR_causal)
    else:
        mu_HR = mu_HR_causal

    mu_HR = torch.nan_to_num(mu_HR, nan=0.0, posinf=0.0, neginf=0.0)

    baseline_t = batch["baseline"][-1].to(DEVICE)
    if baseline_t.dim() == mu_HR.dim() - 1:
        baseline_t = baseline_t.unsqueeze(0)
    baseline_log = torch.nan_to_num(baseline_t, nan=0.0, posinf=0.0, neginf=0.0)

    ensemble = []
    for k in range(K):
        out = diffusion.sample(
            conditioning=None,
            num_steps=n_steps,
            scheduler_type="edm_karras",
            apply_constraints=False,
            mu_HR=mu_HR,
            baseline_log=baseline_log,
        )
        residual = out.residual if hasattr(out, 'residual') else out
        full_pred = baseline_log + mu_HR + residual
        ensemble.append(full_pred.cpu())

    return torch.stack(ensemble, dim=0)

# ── 4. Resolution des interventions ──────────────────────────────
from scripts.intervention_test import (
    INTERVENTIONS, resolve_variable_indices,
    apply_intervention, evaluate_intervention,
    compute_q_int, save_intervention_results,
)
lr_vars = list(CONFIG.data.lr_variables)
resolved = resolve_variable_indices(lr_vars)
print("Interventions resolved :")
for spec in resolved:
    status = "OK" if spec['variable_idx'] is not None else "SKIP"
    print(f"  [{status}] {spec['name']:<32s} {spec['variable_name']:<10s} idx={spec['variable_idx']}")
print()

# ── 5. Boucle d'evaluation ──────────────────────────────────────
N_BATCHES_INT = 2
K_SAMPLES_INT = 4

print(f"Configuration : {N_BATCHES_INT} batches x {K_SAMPLES_INT} K-samples x 3 interventions x 2 variants")
est_seconds = N_BATCHES_INT * K_SAMPLES_INT * 3 * 2 * 2 * 5
print(f"Estimation cout : ~{est_seconds}s ({est_seconds/60:.0f}min)")
print()

t_inf = time.time()
results_v5 = []
results_nc = []

for spec_idx, spec in enumerate(resolved):
    if spec['variable_idx'] is None:
        results_v5.append({"intervention": spec['name'], "skipped": True,
                          "reason": f"variable {spec['variable_name']} absent du LR"})
        results_nc.append({"intervention": spec['name'], "skipped": True,
                          "reason": f"variable {spec['variable_name']} absent du LR"})
        continue

    print(f"[{spec_idx+1}/{len(resolved)}] {spec['name']}")
    deltas_v5 = []
    deltas_nc = []

    sample_iter = iter(test_dataset)
    for batch_idx in range(N_BATCHES_INT):
        try:
            sample = next(sample_iter)
        except StopIteration:
            break
        batch_normal = convert_sample_to_batch(sample, builder, DEVICE)

        pred_normal_v5 = predict_with_stack(stack_v5, batch_normal, K=K_SAMPLES_INT)
        batch_int = dict(batch_normal)
        batch_int['lr'] = apply_intervention(batch_normal['lr'], spec, standardization=None)
        pred_int_v5 = predict_with_stack(stack_v5, batch_int, K=K_SAMPLES_INT)
        deltas_v5.append((pred_int_v5.nanmean(0) - pred_normal_v5.nanmean(0)).mean().item())

        pred_normal_nc = predict_with_stack(stack_nc, batch_normal, K=K_SAMPLES_INT)
        pred_int_nc = predict_with_stack(stack_nc, batch_int, K=K_SAMPLES_INT)
        deltas_nc.append((pred_int_nc.nanmean(0) - pred_normal_nc.nanmean(0)).mean().item())

    delta_v5 = float(np.mean(deltas_v5))
    delta_nc = float(np.mean(deltas_nc))
    sign_pred_v5 = 1 if delta_v5 > 0 else (-1 if delta_v5 < 0 else 0)
    sign_pred_nc = 1 if delta_nc > 0 else (-1 if delta_nc < 0 else 0)
    expected = int(spec['expected_sign'])

    results_v5.append({
        "intervention": spec['name'], "variable": spec['variable_name'],
        "delta_pred_mean": delta_v5,
        "sign_predicted": sign_pred_v5, "sign_expected": expected,
        "match": sign_pred_v5 == expected, "skipped": False,
        "physical_justification": spec['physical_justification'],
    })
    results_nc.append({
        "intervention": spec['name'], "variable": spec['variable_name'],
        "delta_pred_mean": delta_nc,
        "sign_predicted": sign_pred_nc, "sign_expected": expected,
        "match": sign_pred_nc == expected, "skipped": False,
        "physical_justification": spec['physical_justification'],
    })
    print(f"  V5: delta={delta_v5:+.5f} sign={sign_pred_v5:+d} expected={expected:+d} match={'YES' if sign_pred_v5==expected else 'NO'}")
    print(f"  NC: delta={delta_nc:+.5f} sign={sign_pred_nc:+d} expected={expected:+d} match={'YES' if sign_pred_nc==expected else 'NO'}")

print()
print(f"[OK] Phase 8 terminee en {(time.time()-t_inf)/60:.1f} min")
print()

save_intervention_results(results_v5, results_nc, RESULTS_DIR / "phase8_intervention.json")
q_v5 = compute_q_int(results_v5)
q_nc = compute_q_int(results_nc)
print()
print(f"Q_int V5         : {q_v5:.3f}")
print(f"Q_int Noncausal  : {q_nc:.3f}")
print(f"V5 wins intervention : {'YES' if q_v5 > q_nc else 'NO (egal ou perdant)'}")


---

## Synthèse — résultats consolidés

Après exécution des Phases 6, 7, 8, les résultats sont dans :
```
RESULTS_DIR/
├── phase6_in_distribution.json
├── phase7_ood_physical_diagnostics.json (ou phase7_ood.json si Mode A)
└── phase8_intervention.json
```

### Construction du verdict pour la soutenance

**Si V5 gagne in-distribution + OOD + intervention** :
> *« Oracle V5-mini bat le baseline CorrDiff noncausal sur la majorité des métriques in-distribution, dégrade moins sous changement de climat, et satisfait Q_int ≥ 0.9 sur le protocole d'intervention. »*

**Si V5 gagne seulement sur calibration + structure + intervention** (cas trilemme) :
> *« V5-mini affiche un compromis assumé : performance in-distribution proche du noncausal, calibration probabiliste significativement améliorée (+30 %), fidélité spectrale +10 %, et capacité d'intervention démontrée. »*

**Si V5 ne gagne nulle part** :
> *Diagnostic montrant que les pertes V5 ont peut-être été mal calibrées. Plan B : ablation study.*

In [ ]:
# Synthese des resultats sauvegardes
import json

print("=" * 70)
print("SYNTHESE V5 - fichiers generes")
print("=" * 70)
for fname in ["phase6_in_distribution", "phase7_ood_physical_diagnostics", "phase7_ood", "phase8_intervention"]:
    p = RESULTS_DIR / f"{fname}.json"
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  [OK] {p.name:50s}  {size_kb:>8.1f} KB")
    else:
        print(f"  [--] {p.name:50s}  (non genere)")

print()
print("Pour le memoire, voir :")
print("  - Tableau metrique x variant      -> phase6_in_distribution.json (cle 'comparison')")
print("  - D_OOD ou diagnostics physiques  -> phase7_*.json")
print("  - Q_int et signe interventions    -> phase8_intervention.json")